In [ ]:
# Instalasi kebutuhan package
!pip install -q -U google-auth==2.47.0 "google-genai>=1.64.0,<2.0.0" streamlit pyngrok

In [ ]:
# Import komponen
from google.colab import userdata

In [ ]:
# Set token dari colab secret
ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))

In [ ]:
# Aktifasi streamlit dan buka tunnel
from pyngrok import ngrok
import subprocess
import time
def run_streamlit(filename, port=8501):
    # Kill SEMUA proses streamlit, bukan hanya yang kita spawn
    subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
    # Force-free port kalau masih ada yang nempel
    subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)
    # Tutup semua tunnel ngrok
    ngrok.kill()
    # Tunggu port benar-benar bebas
    time.sleep(3)
    # Mulai buka tunnel baru
    proc = subprocess.Popen(
        [
            "streamlit", "run", filename,
            "--server.headless=true",
            "--server.port", str(port),
            "--server.enableCORS=false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    time.sleep(3)
    public_url = ngrok.connect(port)
    print(f"Streamlit berjalan: {public_url}")
    return proc

In [ ]:
#import file document - check apakah file PDF terbaca atau tidak
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
pdf_path = '/content/drive/MyDrive/Colab Notebooks/Hactive8 - AI LLM/project/data.pdf'


Mounted at /content/drive


In [ ]:
# Create application
%%writefile streamlit_chat_app.py
import io
import streamlit as st
from google.genai import types
from google import genai

#konfigurasi AI
system_instruction='Kamu adalah AI untuk membantu menjawab pertanyaan dalam bahasa Indonesia terkait uji kompetensi junior network administrator. Seluruh jawaban yang diberikan harus berdasarkan pengetahuan yang diberikan. Apabila jawaban tidak tersedia, berikan informasi untuk meninggalkan nomor kontak dan pihak sekretariat dan pihak sekretariat akan menghubungi penanya dalam waktu yang secepat-cepatnya melalui email atau telepon.'
chat_config = types.GenerateContentConfig(system_instruction=system_instruction, temperature=2, top_p=0.95, top_k=20)

# 1 - Pemberian judul dan keterangan bagian atas
st.title("Gemini Chatbot")
st.caption("Asisten Tanya Jawab Uji Kompetensi Junior Network Administrator")

# 2 - Pembentukan SideBar
with st.sidebar:
  st.subheader("Pengaturan")
  googgle_key = st.text_input("Google API Key", type="password")
  reset_button = st.button("Mulai Percakapan Baru", help="Hapus catatan percakapan dan mulai dari awal")


# 3 Pengambilan data pada file yang telah disediakan dalam Google Drive
@st.cache_resource(show_spinner="Fetching and analyzing document from Google Drive...")
def load_knowledge_from_drive(file_id, sa_info):
    try:
        # Otentifikasi Google Drive API
        scopes = ['https://googleapis.com']
        creds = service_account.Credentials.from_service_account_info(sa_info, scopes=scopes)
        drive_service = build('drive', 'v3', credentials=creds)
        # Load file PDF ke dalam memory
        request = drive_service.files().get_media(fileId=file_id)
        file_stream = io.BytesIO()
        downloader = MediaIoBaseDownload(file_stream, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
        file_stream.seek(0)
        # Melakukan ekstraksi isi file PDF
        pdf_reader = PyPDF2.PdfReader(file_stream)
        extracted_text = ""
        for page in pdf_reader.pages:
            text = page.extract_text()
            if text:
                extracted_text += text + "\n"
        return extracted_text
    except Exception as e:
        st.error(f"Error loading file from Google Drive: {e}")
        return None

# 4 - Verifikasi Google API Key
if not googgle_key:
    st.info("Masukkan Google AI API Key di sidebar untuk mulai chat.", icon="🗝️")
    st.stop()
    # Pengambilan knowledge yang berada dalam file PDF
    if "gcp_service_account" in st.secrets:
        sa_info = dict(st.secrets["gcp_service_account"])
        document_context = load_knowledge_from_drive('data.pdf', sa_info)
        if document_context:
            st.success("Knowledge base successfully synchronized!")
            if "chat_history" not in st.session_state:
                st.session_state.chat_history = []
            for message in st.session_state.chat_history:
                with st.chat_message(message["role"]):
                    st.markdown(message["content"])
            if user_query := st.chat_input("Pertanyaan yang ingin Anda tanyakan ..."):
                with st.chat_message("user"):
                    st.markdown(user_query)
                st.session_state.chat_history.append({"role": "user", "content": user_query})
                # Konfigurasi prompt embedding
                system_instruction = (
                    "Kamu adalah AI untuk membantu menjawab pertanyaan dalam bahasa Indonesia terkait uji kompetensi junior network administrator."
                    "Seluruh jawaban yang diberikan harus berdasarkan pengetahuan yang diberikan."
                    "Apabila jawaban tidak tersedia, berikan informasi untuk meninggalkan nomor kontak dan pihak sekretariat dan pihak sekretariat akan menghubungi penanya dalam waktu yang secepat-cepatnya melalui email atau telepon.")
                full_prompt = f"Context from Google Drive PDF:\n{document_context}\n\nUser Question: {user_query}"
                # Query Gemini model
                with st.chat_message("assistant"):
                    with st.spinner("Thinking..."):
                        response = ai_client.models.generate_content(
                            model='gemini-2.5-flash',
                            contents=full_prompt,
                            config=genai.types.GenerateContentConfig(
                                system_instruction=system_instruction,
                                temperature=1.3,
                                top_p=0.95,
                                top_k=20)
                        )
                        st.markdown(response.text)
                st.session_state.chat_history.append({"role": "assistant", "content": response.text})
    else:
        st.error("Missing Google Service Account credentials. Please configure `gcp_service_account` in your Streamlit secrets.")

# 5 - Inisialisasi gemini client
if ("genai_client" not in st.session_state) or (getattr(st.session_state, "_last_key", None) != googgle_key):
    try:
        st.session_state.genai_client = genai.Client(api_key=googgle_key)
        st.session_state._last_key = googgle_key
        st.session_state.pop("chat", None)
        st.session_state.pop("messages", None)
    except Exception as e:
        st.error(f"API Key tidak valid: {e}")
        st.stop()

# 6 - Inisialisai cat session dan riwayat chat
if "chat" not in st.session_state:
    st.session_state.chat = st.session_state.genai_client.chats.create(model="gemini-2.5-flash", config=chat_config)
if "messages" not in st.session_state:
    st.session_state.messages = []

# 7 - Event handler tombol reset
if reset_button:
    st.session_state.pop("chat", None)
    st.session_state.pop("messages", None)
    st.rerun()

# 8 - Riwayat percakapan
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# 9 - Input dan respon
prompt = st.chat_input("Ketik pertanyaanmu di sini...")
if prompt:
    st.session_state.messages.append({"role": "user", "content": prompt})       # Langkah 1: Tambah pesan user ke riwayat
    with st.chat_message("user"):                                               # Langkah 2: Tampilkan bubble pesan user
        st.markdown(prompt)
    try:                                                                        # Langkah 3: Kirim ke Gemini dan tampilkan respons
        response = st.session_state.chat.send_message(prompt)
        if hasattr(response, "text"):
            answer = response.text
        else:
            answer = str(response)
    except Exception as e:
        answer = f"Terjadi error: {e}"
    with st.chat_message("assistant"):                                          # Langkah 4: Tampilkan bubble respons assistant
        st.markdown(answer)
    st.session_state.messages.append({"role": "assistant", "content": answer})  # Langkah 5: Simpan respons ke riwayat

Overwriting streamlit_chat_app.py


In [ ]:
# Jalankan applikasi
proc = run_streamlit("streamlit_chat_app.py")
print("Aplikasi telah dijalankan.")

Streamlit berjalan: NgrokTunnel: "https://shrewdly-rack-shrug.ngrok-free.dev" -> "http://localhost:8501"
Aplikasi telah dijalankan.


In [ ]:
# Stop aplikasi streamlit
try:
    proc.terminate()
    print("Aplikasi dihentikan.")
except:
    print("Aplikasi belum pernah dijalankan.")

# Tutup semua tunnel ngrok
ngrok.kill()
print("Aplikasi berhasil ditutup.")

Aplikasi dihentikan.
Aplikasi berhasil ditutup.
